# 01 - Qdrant desde cero

Objetivo: entender `Qdrant` como componente aislado, sin LangChain y sin pipeline RAG.


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"
COLLECTION_NAME = "sandbox_manual_vectors"

client = QdrantClient(url=QDRANT_URL)
print({"QDRANT_URL": QDRANT_URL, "COLLECTION_NAME": COLLECTION_NAME})


{'QDRANT_URL': 'http://127.0.0.1:6333', 'COLLECTION_NAME': 'sandbox_manual_vectors'}


In [2]:
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)

points = [
    PointStruct(id=1, vector=[0.9, 0.1, 0.0, 0.0], payload={"label": "devoluciones"}),
    PointStruct(id=2, vector=[0.0, 0.9, 0.1, 0.0], payload={"label": "montaje"}),
    PointStruct(id=3, vector=[0.0, 0.1, 0.9, 0.2], payload={"label": "envios"}),
]

client.upsert(collection_name=COLLECTION_NAME, points=points)
client.get_collection(COLLECTION_NAME)


C:\Users\anton\AppData\Local\Temp\ipykernel_23256\1093027847.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=3, segments_count=4, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=4, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), w

In [3]:
query_vector = [0.85, 0.15, 0.0, 0.0]
hits = client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=2).points

for hit in hits:
    print({
        "id": hit.id,
        "score": round(hit.score, 4),
        "payload": hit.payload,
    })


{'id': 1, 'score': 0.998, 'payload': {'label': 'devoluciones'}}
{'id': 2, 'score': 0.1727, 'payload': {'label': 'montaje'}}
